# 03. Temporal Fact Retrieval Pipeline Demo

Demonstration of the **Temporal Fact Retrieval** subsystem:
1. **Query Parsing**: Entity linking via fuzzy & known-entity resolution, plus regex temporal extraction.
2. **Template Selection**: Cypher/Query dispatch (`entity_at_time`, `entity_pair`, `events_before`, `events_after`).
3. **Relevance Ranking**: Temporal proximity scoring and context formatting.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config import settings
from src.knowledge_graph.memory_store import InMemoryKGStore
from src.retrieval.query_parser import QueryParser
from src.retrieval.temporal_retriever import TemporalRetriever

## 1. Initializing Query Parser & Graph Store

In [ ]:
store = InMemoryKGStore()
store.load(settings.PROCESSED_DATA_DIR / "knowledge_graph.json")

entity_names = store.get_all_entity_names()
parser = QueryParser(known_entities=entity_names)
retriever = TemporalRetriever(graph_store=store, query_parser=parser, top_k=5)
print(f"Loaded retriever with {len(entity_names)} known entities.")

## 2. Natural Language Query Parsing

In [ ]:
test_queries = [
    "Who did the United States make statements to in March 2014?",
    "What events occurred between United States and Russia in 2014?",
    "What happened before June 2014 regarding China?",
    "What events occurred after October 2014 involving Germany?"
]

for query in test_queries:
    parsed = parser.parse(query)
    print(f"Query: {query}")
    print(f"  Query Type:  {parsed['query_type']}")
    print(f"  Entities:    {parsed['entities']}")
    print(f"  Date Range:  {parsed['date_range']}")
    print("-" * 60)

## 3. End-to-End Temporal Fact Retrieval

In [ ]:
sample_q = "What events occurred between United States and Russia in March 2014?"
facts = retriever.retrieve(sample_q, top_k=5)

print(f"Retrieved {len(facts)} facts for: '{sample_q}':")
for i, f in enumerate(facts, 1):
    print(f"{i}. [{f['timestamp']}] ({f['subject']}) --[{f['predicate']}]--> ({f['object']}) [Relevance: {f['relevance_score']:.2f}]")